In [1]:
import warnings
warnings.filterwarnings("ignore")
import logging
# Suppress INFO logs, show only WARNING and above
logging.getLogger().setLevel(logging.WARNING)


import numpy as np
import scanpy as sc
from repspat import SampleData, spatial_silhouette_analysis,spatial_constrained_hac, plot_spatial_clusters, plot_cluster_feature_presence, pairwise_results_to_matrix, multiple_comparison, create_blocks


In [2]:
binary_layer = "binary"
thresholds = {
    "betaCatenin": 0.253063192,
    "CD11b": 1.100536222,
    "CD11c": 0.213556566,
    "CD138": 1.038502303,
    "CD16": 0.304460727,
    "CD20": 0.296460556,
    "CD209": 0.406669859,
    "CD3": 0.250762424,
    "CD31": 1.006381212,
    "CD4": 0.223762,
    "CD45": 0.327450495,
    "CD45RO": 0.236334141,
    "CD56": 0.770183826,
    "CD63": 0.268126101,
    "CD68": 1.128510101,
    "CD8": 0.260237333,
    "dsDNA": 0.20301697,
    "EGFR": 1.069264889,
    "FoxP3": 0.34649899,
    "H3K27me3": 0.201893232,
    "H3K9ac": 0.406071229,
    "HLA_Class_1": 0.23773137,
    "HLADR": 0.4408116,
    "IDO": 0.312824202,
    "Keratin17": 0.291843697,
    "Keratin6": 0.282480556,
    "Ki67": 0.210413737,
    "Lag3": 1.239097091,
    "MPO": 0.324934545,
    "p53": 0.265704293,
    "panKeratin": 0.244129571,
    "PD1": 0.972859909,
    "PDL1": 1.352024101,
    "pS6": 0.282699091,
    "SMA": 0.250358024,
    "Vimentin": 0.28239596,
}

In [3]:
adata = sc.read_h5ad('../data/03_TNBC_2018_spe.h5ad')
missing = [marker for marker in adata.var_names if marker not in thresholds]
if missing:
    raise ValueError(f"Missing thresholds for: {missing}")
X = adata.X.toarray() if hasattr(adata.X, "toarray") else adata.X
cutoffs = np.array([thresholds[marker] for marker in adata.var_names])
adata.layers[binary_layer] = (X >= cutoffs).astype(int)


In [4]:
data = SampleData(adata, sample_column="sample_id", sample_name="Sample_04", layer=binary_layer, metric="jaccard")
sample = data.sample_adata

In [5]:
data.summary()

{'feature_mat': (5381, 36),
 'coords_mat': (5381, 2),
 'dist_matrix': (5381, 5381),
 'cell_type': (5381, 1),
 'sample_adata': (5381, 36)}

In [7]:
data.feature_mat

,betaCatenin,CD11b,CD11c,CD138,CD16,CD20,CD209,CD3,CD31,CD4,...,Ki67,Lag3,MPO,p53,panKeratin,PD1,PDL1,pS6,SMA,Vimentin
Cell_1405,0,0,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,1,1,1
Cell_1406,0,0,1,0,0,1,0,1,0,1,...,0,0,0,0,0,0,0,1,0,1
Cell_1407,0,0,0,0,0,0,0,0,0,0,...,1,0,0,0,0,0,0,1,0,0
Cell_1408,0,0,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,1,0,1
Cell_1409,0,0,1,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,1,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Cell_178418,0,0,1,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,1,0,0
Cell_178419,0,0,1,0,0,0,0,1,0,1,...,0,0,0,0,0,0,0,1,0,1
Cell_178420,1,0,0,0,0,0,0,1,0,1,...,0,0,0,0,0,0,0,0,0,1
Cell_178421,0,0,0,0,0,0,0,1,0,1,...,0,0,0,0,0,0,0,1,0,1


In [ ]:
# Match the previous workflow: keep binary/Jaccard distances,
# but run Ward clustering on raw sample.X.
binary_layer_for_features = sample.uns["repspat"]["layer"]
sample.uns["repspat"]["layer"] = None

spatial_silhouette_analysis(
    sample,
    n_neighbors_list=[6, 8],
    n_clusters_range=range(4, 9),
    linkage="ward",
)


In [ ]:
labels, sample, model = spatial_constrained_hac(
    sample,
    n_clusters=7,
    n_neighs=8,
    linkage="ward",
)

sample.uns["repspat"]["layer"] = binary_layer_for_features


In [ ]:
coords = sample.obsm["spatial"]
x = coords[:, 0]
y = coords[:, 1]

In [ ]:
plot_spatial_clusters(sample,point_size=5)

In [ ]:
cluster_feature_plots = plot_cluster_feature_presence(sample, top_n=10, point_size=2)

In [ ]:
create_blocks(sample, knn=8)

In [ ]:
results_df = multiple_comparison(sample, kernel='IMQ')

In [ ]:
results_df[results_df['adj_p'] >= 0.05]

In [ ]:
pairwise_results_to_matrix(sample)